## Preliminares

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer
import sklearn.impute as skl_imp
from sklearn.experimental import enable_iterative_imputer # Necesario para usar skl_imp, no borrar
from src.config import data_folder
from src.transform import *

In [2]:
# Abrir archivo raw_data
df = pd.read_parquet(f"{data_folder}/raw_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11372 entries, 0 to 11371
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         11372 non-null  datetime64[ns]
 1   Ticker                       11372 non-null  object        
 2   Close                        11372 non-null  float64       
 3   Open                         11372 non-null  float64       
 4   Volume                       11372 non-null  float64       
 5   DateAdded                    6705 non-null   object        
 6   Sector                       11372 non-null  object        
 7   Industry                     11372 non-null  object        
 8   TotalRevenue                 11372 non-null  float64       
 9   GrossProfit                  10519 non-null  float64       
 10  OperatingIncome              11372 non-null  float64       
 11  NetIncome                    11372 non-nu

* Para mejorar la visualización de los datos, se expresan las columnas financieras y volumen en miles:

In [3]:
df = financieras_en_millones(df)

In [4]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,11372,2023-08-23 23:59:06.816743168,2020-09-01 00:00:00,2022-03-01 00:00:00,2023-09-01 00:00:00,2025-03-01 00:00:00,2026-06-01 00:00:00,NaN
Close,11372.0,1040.619655,0.57,41.017641,82.728516,165.279922,775000.0,22865.080943
Open,11372.0,1008.193786,0.53,40.525034,81.592607,161.329193,775648.0,22114.560371
Volume,11372.0,329.204772,0.0002,44.420575,103.52325,258.60355,36280.458,1251.844401
TotalRevenue,11372.0,6593.381045,-12134.0,1275.91075,2386.5,5278.8,227173.0,14899.692411
GrossProfit,10519.0,2361.834557,-24353.0,422.4545,831.622,1828.5,137192.0,6375.388808
OperatingIncome,11372.0,967.03061,-55512.0,125.3455,303.581,820.05,77648.0,2944.934098
NetIncome,11372.0,718.475912,-43621.0,65.47875,199.0,581.0,62578.0,2672.06322
EBITDA,1885.0,1561.779546,-16168.444,197.767,466.778,1202.0,84427.0,5209.528381
BasicAverageShares,11326.0,641.974296,0.00021,95.826629,219.801,550.57775,30864.0,1743.662643


## Corrección de errores

Anomalías de signo: 

Las siguientes variables no pueden ser negativas:
* `TotalRevenue`
* `CurrentDebt`
* `LongTermDebt`
* `DepreciationAndAmortization`

In [5]:
# Visualizar casos de TotalRevenue negativo
condicion = df['TotalRevenue'] < 0
cols_a_visualizar = ['Ticker', 'Date', 'TotalRevenue', 'OperatingIncome']
anomalias = df.loc[condicion, cols_a_visualizar]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

      Ticker       Date  TotalRevenue  OperatingIncome
414      ALB 2021-03-01   -1790.31392      -114.758843
955     AVNT 2021-03-01     -30.20000       -12.700000
1182     BAX 2023-03-01    -704.00000      -285.000000
1286     BHF 2022-03-01    -150.00000      2266.000000
1290     BHF 2023-03-01    -127.00000        40.000000
2279     CNA 2023-03-01   -3542.00000     -4648.000000
2901     DAN 2024-03-01    -448.00000      -387.000000
3140    DLTR 2024-06-01   -5182.80000       765.300000
3531     EMR 2021-12-01    -357.00000      -206.000000
4096    FTNT 2022-03-01   -1415.00000       213.700000
4126     FTV 2024-03-01    -567.70000      -248.800000
4217      GE 2023-03-01  -12134.00000      4010.000000
4838     HLT 2024-03-01   -3218.00000       438.000000
5485       J 2022-12-01   -1258.70300       -69.426000
5489       J 2023-12-01   -1212.28200      -120.924000
6405     MAR 2024-03-01  -11318.00000       726.000000
6409     MAR 2025-03-01  -12053.00000       804.000000
7190    ND

* No se puede concluir que se traten de alteraciones de signo en el parseo de datos de simFin. Se observa en 2 casos que el valor absoluto de `TotalRevenue` es mayor que `OperatingIncome`, es imposible que los resultados sean mayores que las ventas.

Se opta por asignar todos estos valores anómalos a NaN.

In [6]:
df['TotalRevenue'] = np.where(df['TotalRevenue'] < 0, np.nan, df['TotalRevenue'])

* Casos de deuda negativa: se calcula "PasivoImplicito", que surge de aplicar la ecuación contable fundamental `Activo` = `Pasivo` + `Patrimonio Neto`

In [7]:
#se crea "PasivoImplicito"
df['PasivoImplicito'] = df['TotalAssets'] - df['StockholdersEquity']

condicion = (df['CurrentDebt'] < 0) | (df['LongTermDebt'] < 0)
cols_a_visualizar = ['Ticker', 'Date', 'TotalDebt', 'CurrentDebt', 'LongTermDebt', 'PasivoImplicito']
anomalias = df.loc[condicion, cols_a_visualizar]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

      Ticker       Date  TotalDebt  CurrentDebt  LongTermDebt  PasivoImplicito
5139     IEP 2022-09-01        NaN     -746.000      7134.000        15991.000
5140     IEP 2022-12-01        NaN     -745.000      7127.000        16908.000
9449    STLD 2024-03-01        NaN      459.987     -3286.537          171.287
9450    STLD 2024-06-01        NaN      425.696     -3570.028         -203.283
9452    STLD 2024-12-01        NaN      882.013     -3997.348         -218.698
10233    TXN 2025-03-01        NaN      750.000    -28049.000       -22289.000
Cantidad de casos: 6


* Se observa que la ecuación contable fundamental no se cumple. 

Se decide eliminar estos registros "tóxicos" del dataset.

In [8]:
condicion = (df['CurrentDebt'] < 0) | (df['LongTermDebt'] < 0)
df = df[~condicion].reset_index(drop=True)
df = df.drop(columns= 'PasivoImplicito')

* Casos de negativos en `DepreciationAndAmortization`: 

se analizan los casos considerando la ecuación `EBITDA` = `OperatingIncome` + `DepreciationAndAmortization`

In [9]:
# Casos de negativos en "DepreciationAndAmortization"
condicion = df['DepreciationAndAmortization'] < 0
cols_a_visualizar = ['Ticker', 'Date', 'DepreciationAndAmortization', 'EBITDA', 'OperatingIncome']
anomalias = df.loc[condicion, cols_a_visualizar]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

      Ticker       Date  DepreciationAndAmortization  EBITDA  OperatingIncome
19        AA 2020-12-01                       -161.0     NaN            113.0
20        AA 2021-03-01                       -170.0     NaN            184.0
21        AA 2021-06-01                       -182.0     NaN            337.0
22        AA 2021-09-01                       -161.0     NaN            456.0
23        AA 2021-12-01                       -156.0     NaN            570.0
...      ...        ...                          ...     ...              ...
11357    ZTS 2024-06-01                        -37.0     NaN            801.0
11358    ZTS 2024-09-01                        -35.0     NaN            906.0
11359    ZTS 2024-12-01                        -35.0     NaN            920.0
11360    ZTS 2025-03-01                        -34.0     NaN            765.0
11361    ZTS 2025-06-01                        -32.0     NaN            812.0

[4262 rows x 5 columns]
Cantidad de casos: 4262


* Los datos que provienen de `simFin` utilizan signo negativo para esta columna, lo cual es incorrecto. 

* Dichos valores serán convertidos a positivos.

A continuación se verifican si hay datos negativos que vengan de yfinance (con EBITDA):

In [10]:
condicion = (df['DepreciationAndAmortization'] < 0) & (df['EBITDA'].notna())
cols_a_visualizar = ['Ticker', 'Date', 'DepreciationAndAmortization', 'EBITDA', 'OperatingIncome']
anomalias = df.loc[condicion, cols_a_visualizar]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

     Ticker       Date  DepreciationAndAmortization  EBITDA  OperatingIncome
4408    GME 2026-06-01                         -0.3   144.9            144.9
Cantidad de casos: 1


* En este caso se observa que el valor implicito de `DepreciationAndAmortization` es igual a cero. El valor de -0.3 pudo haberse tratado de un pequeño "ajuste contable", $300.000 dólares para una compañia del tamaño de `GameStop` es contablemente irrelevante.

Se reemplaza dicho valor por cero:

In [11]:
df.loc[4408,'DepreciationAndAmortization'] = 0

In [12]:
# Ahora se convierten a positivos los restantes
df['DepreciationAndAmortization'] = df['DepreciationAndAmortization'].abs()

# Verificar cambios
condicion = df['DepreciationAndAmortization'] < 0
cols_a_visualizar = ['Ticker', 'Date', 'DepreciationAndAmortization', 'EBITDA', 'OperatingIncome']
anomalias = df.loc[condicion, cols_a_visualizar]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

Empty DataFrame
Columns: [Ticker, Date, DepreciationAndAmortization, EBITDA, OperatingIncome]
Index: []
Cantidad de casos: 0


In [13]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,11366,2023-08-23 22:09:00.981875712,2020-09-01 00:00:00,2022-03-01 00:00:00,2023-09-01 00:00:00,2025-03-01 00:00:00,2026-06-01 00:00:00,NaN
Close,11366.0,1041.115928,0.57,41.025952,82.713306,165.299973,775000.0,22871.10556
Open,11366.0,1008.669525,0.53,40.52694,81.584925,161.399801,775648.0,22120.3874
Volume,11366.0,329.305029,0.0002,44.441125,103.55245,258.71585,36280.458,1252.160723
TotalRevenue,11341.0,6615.658959,0.756217,1278.859,2394.271,5297.2,227173.0,14912.315086
GrossProfit,10513.0,2362.674765,-24353.0,422.484,832.2,1829.0,137192.0,6377.090679
OperatingIncome,11366.0,967.292552,-55512.0,125.35625,303.5175,820.15,77648.0,2945.66569
NetIncome,11366.0,718.65458,-43621.0,65.491,199.0,581.0,62578.0,2672.73675
EBITDA,1885.0,1561.779546,-16168.444,197.767,466.778,1202.0,84427.0,5209.528381
BasicAverageShares,11320.0,642.136371,0.00021,95.762879,219.801,550.64475,30864.0,1744.099643


* Se imputan parte de los NaNs en variables de Deuda antes de calcular métricas, 
mediante las relaciones contables entre ellas:

In [ ]:
df_debt_imputed = imputar_deuda(df)
mostrar_missings(df_debt_imputed)

* Se crea feature `YearsSinceAdded`:

In [ ]:
#  Pasar DateAdded a formato datetime, los NaN se vuelven NaT (not a time)
df_debt_imputed['DateAdded'] = pd.to_datetime(df_debt_imputed['DateAdded'], errors='coerce')
# Convertir a YearsSinceAdded, aqui los nulos regresan a NaN
df_debt_imputed['YearsSinceAdded'] = round(((pd.Timestamp.now() - df_debt_imputed['DateAdded']).dt.days / 365.25), 0)

# 3. Se asignan a cero años los tickers que no pertenecen al Índice S&P 500
df_debt_imputed['YearsSinceAdded'] = df_debt_imputed['YearsSinceAdded'].fillna(0)

# Eliminar la columna original
df_debt_imputed.drop('DateAdded', axis=1, inplace=True)
mostrar_missings(df_debt_imputed)

* Se aplica forward fill para cubrir posibles huecos de hasta 1 período:

In [ ]:
df_debt_imputed = cubrir_huecos(df_debt_imputed,limite=1)
mostrar_missings(df_debt_imputed)

* Se convierten las variables de flujo trimestrales a valores TTM (ventana móvil de 4 trimestres):

In [ ]:
df_debt_imputed = transformar_flujos_a_ttm(df_debt_imputed)
mostrar_missings(df_debt_imputed)

* Calcular métricas:

In [ ]:
df_with_metrics, crecimiento_cols = calcular_metricas(df_debt_imputed)
mostrar_missings(df_with_metrics)

* Se aplica imputación transversal para las columnas de crecimiento:

In [ ]:
df_with_metrics = imputar_transversal(df_with_metrics, crecimiento_cols)
mostrar_missings(df_with_metrics)

* Calcular los retornos trimestrales, varianza del activo y covarianza con el mercado para cada ticker:

In [ ]:
# Se abre el fichero de precios del Índice del Mercado
df_index = pd.read_parquet(f"{data_folder}/market_index.parquet")
df_with_features = calcular_retornos(df_with_metrics, df_index)
mostrar_missings(df_with_features)

In [ ]:
df_with_features.describe().T

## Missing Values

In [ ]:
# Se separan columnas numéricas y no numéricas
df_cont = df_with_features.select_dtypes(include='number')
df_non_numeric = df_with_features.select_dtypes(exclude='number')

# Comprobar valores infinitos antes de imputación multivariable
print(np.isinf(df_cont).sum())

In [ ]:
# NaN Restantes: Imputación multivariable con IterativeImputer sobre numéricas
# Imputador: Chain equations
imputer_itImp = skl_imp.IterativeImputer(max_iter=10, random_state=0)

df_cont_imputed = pd.DataFrame(imputer_itImp.fit_transform(df_cont),columns=df_cont.columns)
mostrar_missings(df_cont_imputed)

In [ ]:
# Se vuelven a unir las columnas numéricas y no numéricas
df_imputed = pd.concat([df_cont_imputed, df_non_numeric], axis=1)

## Transformaciones

In [ ]:
# Se calculan tamaños relativos: RelativeAssets y RelativeRevenue
df_transformed = calcular_relative_size(df_imputed)
df_transformed.info()

In [ ]:
# Se expresan columnas de capitalización y de mercado en billions
cols = [
    'MarketCap', 
    'EnterpriseValue', 
    'TotalMarketAssets', 
    'TotalMarketRevenue'
    ]

for col in cols:
    df_transformed[col] = df_transformed[col] / 10**9

In [ ]:
# Convertir Sector y Industry a tipo category
df_transformed['Sector'] = df_transformed['Sector'].astype('category')
df_transformed['Industry'] = df_transformed['Industry'].astype('category')

# Valores unicos en Sector
df_transformed['Sector'].value_counts()

In [ ]:
# Valores unicos en Industry
df_transformed['Industry'].value_counts()

In [ ]:
# Distribución de variables contínuas
df_transformed.describe().round(4).T

In [ ]:
# Coeficientes de asimetría
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Graficar
columna_a_graficar = 'NetDebtToEbitda' # indicar columna para el grafico
plot(df_transformed[columna_a_graficar])

In [ ]:
# Transformaciones logarítmicas

columnas_a_transformar = [ 
    'RelativeAssets',
    'RelativeRevenue'
    ]
for columna in columnas_a_transformar:
    df_transformed[columna] = df_transformed[columna].fillna(0)
    df_transformed[f'{columna}_log'] = np.log1p(df_transformed[columna])
    df_transformed.drop(columna, axis=1, inplace=True)

# Coeficientes de asimetria actualizado
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Definir columnas que saltean la "winsorización"
columnas_intactas = cols_monetarias + [
    # Variables de precio (posibles label)
    'Close',
    'Open',    
    'TrailingPE',
    'EnterpriseToEbitda',
    'PriceToBook',
    # Otras
    'Date', 
    'Ticker'       
    ]

# Separar el dataset
df_passthrough = df_transformed[columnas_intactas].copy()
df_transformed_features = df_transformed.drop(columns=columnas_intactas)

## Gestión de Outliers

Se winsorizan los valores atipicos en las variables continuas que cumplan los siguientes criterios:

Para variables simetricas:
* A mas de 3 desviaciones tipicas de la media.
* Mas de 3 rangos intercuartilicos.

Para variables asimetricas (modulo del coeficiente de asimetrica mayor a 1):
* A mas de 3 MADs de la mediana.
* Mas de 3 rangos intercuartilicos.

In [ ]:
# Outliers
df_cont_transformed = df_transformed_features.select_dtypes(include="number")
df_winsor = df_cont_transformed.apply(lambda x: gestiona_outliers(x, clas='winsor'))

In [ ]:
# Coeficientes de asimetria luego de winsorizar
df_winsor.skew().sort_values(ascending=False)

In [ ]:
# Visualizar cambios
columna_a_graficar = 'RelativeAssets_log' # indicar columna para el grafico
plot(df_winsor[columna_a_graficar])

In [ ]:
df_winsor.describe().T

## Concatenación final de columnas

In [ ]:
df_non_numeric_transformed = df_transformed_features.select_dtypes(exclude='number')
# Se unen variables contínuas transformadas y variables no numéricas
df_combined = pd.concat([df_non_numeric_transformed, df_winsor], axis=1)

# Unir con las columnas que fueron salteadas
df_final = pd.concat([df_passthrough, df_combined], axis=1)
df_final.info()

In [ ]:
# Guardar datos extraidos en fichero clean_data
df_final.to_parquet(f"{data_folder}/clean_data.parquet")